# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Published Date: {metadata.datePublished}\n")

print("Available keywords:")
pprint.pprint(metadata.keywords)

print("\nAvailable data use cases:")
pprint.pprint(metadata.dataUseCases)

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets with their `@id` and field details
record_sets = list(dataset.record_sets)

print("Available Record Sets in the Dataset (by @id):\n")
for rs in record_sets:
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}")
        print(f"      @id: {field.id}")
        print(f"      dataType: {getattr(field, 'dataType', 'N/A')}")
        print(f"      Description: {field.description}")
    print("")
# Save first record set ID for downstream examples
if len(record_sets) > 0:
    main_record_set_id = record_sets[0].id
    print(f"\nPrimary record set chosen for analysis: {main_record_set_id}")
else:
    print("No record sets found in Croissant schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set (by @id)
dataframes = {}

for rs in record_sets:
    print(f"Loading records for record set: {rs.name} (@id: {rs.id}) ...")
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"  Data shape: {df.shape}")
        print(f"  Fields: {df.columns.tolist()}")
    except Exception as e:
        print(f"  Failed to load records: {e}")

# Preview the head() of the main record set DataFrame
if main_record_set_id in dataframes:
    print(f"\nHead of primary DataFrame ({main_record_set_id}):")
    display(dataframes[main_record_set_id].head())
else:
    print(f"No DataFrame loaded for {main_record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Find numeric fields for the selected record set (@id)
df = dataframes.get(main_record_set_id)
if df is not None:
    print("Data types detected in DataFrame:")
    print(df.dtypes)

    # Try to find a numeric field (float/int) for demonstration
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if len(numeric_fields) == 0:
        # Attempt to convert plausible columns to numeric
        plausible_numeric = [col for col in df.columns if any(x in col.lower() for x in ["count", "coverage", "mw", "mass", "abundance", "pi"])]
        for col in plausible_numeric:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if len(numeric_fields) == 0:
        print("No numeric fields found for EDA.")
    else:
        numeric_field = numeric_fields[0]
        print(f"Selected numeric field for demonstration: {numeric_field}")

        # Filtering records where numeric_field > threshold
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df[[numeric_field]].head())

        # Normalizing the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

        # Try to group by a categorical field
        # We'll choose the first column that is not numeric or the 'accession' or 'description' column
        group_field_candidates = [col for col in df.columns if (not pd.api.types.is_numeric_dtype(df[col])) and (col.lower() != numeric_field.lower())]
        group_field = None
        if "description" in df.columns:
            group_field = "description"
        elif len(group_field_candidates) > 0:
            group_field = group_field_candidates[0]
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (showing first 5 groups):")
            print(grouped_df.head())
else:
    print("Primary DataFrame not available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and len(numeric_fields) > 0:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field is not None:
        # Plot mean of numeric_field per group (e.g. protein description or accession)
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10, 6))
        sns.barplot(x=group_means.values, y=group_means.index)
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* This notebook demonstrates how to use the `mlcroissant` library to explore a FAIR dataset described using a Croissant schema.
* We loaded and examined the metadata, record sets (referenced by their `@id`), and sampled data fields.
* Sample exploratory data analysis included filtering, normalization, and grouping, followed by basic visualizations.
* For further analysis, users can choose alternate record sets or fields based on the detailed schema overview cells above.